# 08 · Portfolio application: asset-level bounds in a portfolio risk budget

Additional analysis requested in peer feedback. It **only reads saved forecasts and bounds**; nothing from
notebooks 01-07 is changed. All settings are fixed in the header of `scripts/portfolio_risk_budget.py` and must not be
changed after seeing the results.

Monthly horizon, DLinear base. Each month three long-only portfolios are formed (equal weight, minimum variance, and
minimum variance with the upper-bound covariance). Exposure is scaled so that a portfolio volatility bound built from the
asset-level upper bounds equals a 10% annual budget. Two portfolio bounds are compared:
* **additive**: $\sum_i w_i\sigma_i^{up}$, never exceeded when every instrument is covered (no assumption on dependence);
* **dependence-adjusted**: $\sqrt{w^\top D C D w}$ with the estimated correlation $C$ (no guarantee).

Keep `config/config.yaml` exactly as in the test-period run (`mode: holdout`). CPU runtime is enough (about 5-10 minutes).

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

## Locate the script and the saved bounds

In [ ]:
import shutil
from pathlib import Path
SCRIPTS = Path(PROJECT_DIR) / "scripts"
SCRIPTS.mkdir(exist_ok=True)
name = "portfolio_risk_budget.py"
script = None
for loc in [SCRIPTS / name, Path(PROJECT_DIR) / "src" / "calvol" / name, Path(PROJECT_DIR) / name]:
    if loc.exists():
        if loc.parent != SCRIPTS:
            shutil.copy(loc, SCRIPTS / name)
            print(f"copied {loc.relative_to(PROJECT_DIR)} -> scripts/{name}")
        script = SCRIPTS / name
        break
assert script is not None, "Place portfolio_risk_budget.py in scripts/ or src/calvol/"
assert MODE == "holdout", "Run this notebook with the same config as the test-period run (mode: holdout)"
for f in [P.processed / "assets.npz", P.bounds / "assets_h21.parquet"]:
    print(("OK      " if f.exists() else "MISSING ") + str(f))

## Run

In [ ]:
p = subprocess.Popen([sys.executable, str(script)], cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
assert p.returncode == 0, "script failed (see output above)"

### Checks
The recomputed targets must match the saved targets (relative difference below about 1e-5), and the Minkowski slack must never be negative.

In [ ]:
c = pd.read_csv(P.tables / "portfolio_risk_budget_checks.csv")
print("max relative target difference:", c["max_abs_rel_diff_target"].max())
print("smallest Minkowski slack:", c.filter(like="minkowski").min().min())
assert c["max_abs_rel_diff_target"].max() < 1e-4
assert c.filter(like="minkowski").min().min() >= -1e-12

## Files to send
The archive contains the new tables only.

In [ ]:
import zipfile
z = P.results_root / "calvol_portfolio_results.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(P.tables.glob("portfolio_risk_budget*")):
        zf.write(f, f.relative_to(P.results_root))
        print("added", f.name)
print("Archive:", z)
if IN_COLAB:
    from google.colab import files
    files.download(str(z))